# ✅ Solutions — Chapter 9 — Ontologies and Natural Languages — agentic lab

This is the **solution** notebook: the same lab with all 3 tasks worked. It runs clean end to end, which is what proves the reference implementations satisfy the marking scheme.

> Student version: [`04_agentic_lab.ipynb`](04_agentic_lab.ipynb)

# Chapter 9 — Ontologies and Natural Languages
### Notebook 4 · Agentic lab — verbalising, and when to stop resampling

*Book reference: Extends §9.1–9.2*

A task with a free exact grader, a second half that the exact grader cannot judge, and the course's first **optimal-stopping** MDP — which is best-of-n sampling with the arithmetic done.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch09_toolkit as ch9
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import ch09_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

**By the end of this notebook you can:**

1. Build an agent graded by a **function**, with no gold labels anywhere.
2. Score fidelity and presentation separately, and see an agent max one while failing the other.
3. Derive the **stopping rule** for best-of-n sampling by value iteration.
4. Explain why the threshold falls as the budget runs out.

> **Prerequisite:** the Chapter 1 agentic lab.

## 1. Tools

`check_round_trip` is the unusual one: it is a **grader the agent can call on itself**. Most tasks in this course require a gold answer to know whether the output is right; here the agent can find out unaided, before submitting.

In [ ]:
ctx = AG.Ch9Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:20s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":20s} {t.description.splitlines()[0]}')

In [ ]:
print(tools['lookup_label'].invoke({'term': 'Herbivore', 'language': 'de'}))
print(tools['check_round_trip'].invoke(
    {'sentence': 'Every lion eats at least one herbivore.', 'language': 'en'}))
print(tools['check_round_trip'].invoke(
    {'sentence': 'Lions tend to eat herbivores.', 'language': 'en'}))
print(tools['check_readability'].invoke({'sentence': 'No plant is a animal.'}))
print('\ntrajectory:', ctx.log.names())

## 2. The dataset and the two-part metric

Every sample axiom in every language — 24 examples, stratified on `(construct, language)` so both halves see every construct in every language.

The score is **half fidelity, half presentation**:

| half | measured by | kind |
|---|---|---|
| does it mean the right thing? | round trip | exact decision procedure |
| would anyone read it? | readability + lexicon check | proxy for a judge |

In [ ]:
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print(f'train {len(train)}, dev {len(dev)}')
print('train constructs:', sorted({e.operator for e in train}))
print('dev   constructs:', sorted({e.operator for e in dev}))
print('train languages :', sorted({e.language for e in train}))
print('dev   languages :', sorted({e.language for e in dev}))

In [ ]:
lm = llm.configure_dspy(AG.CNL_RULEBOOK, AG.cnl_responder)
baseline = AG.VerbalisationProgram()
example = dev[0]
pred = baseline(**example.inputs())
print('axiom    :', example.axiom, f'({example.language})')
print('produced :', repr(pred.sentence))
report = AG.verbalisation_scorer(example, pred)
print('score    :', report.score)
for n in report.notes:
    print('   ', n)

In [ ]:
before = ev.evaluate_dataset(baseline, dev, AG.verbalisation_scorer)
print('BEFORE:', before['mean_score'])
print('violations:', before['violations'])

## 3. GEPA

Each construct has its **own** rule, so each discovery pays off immediately. That is a deliberate design choice: an earlier draft gated the templates behind a single meta-rule, and improvement stayed invisible until *two* rules were found together — a credit-assignment trap that stalled the optimiser completely.

In [ ]:
gepa_metric = ev.make_gepa_metric(AG.verbalisation_scorer, AG.CNL_RULEBOOK)
reflect = llm.reflection_lm(AG.CNL_RULEBOOK, AG.cnl_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=180, reflection_lm=reflect)
result = opt.compare(AG.VerbalisationProgram(), tuned, dev, AG.verbalisation_scorer)
print(result.report()[:1500])

In [ ]:
found = AG.CNL_RULEBOOK.active_in(result.instruction_after)
print(f'rules discovered: {len(found)}/{len(AG.CNL_RULEBOOK.ids)}')
print('missed:', sorted(set(AG.CNL_RULEBOOK.ids) - found) or 'none')

### The rule the exact metric could not have taught

`use-the-lexicon-label` is worth dwelling on. An agent that uses English identifiers in a Dutch sentence still **round-trips perfectly** — the parser falls back to returning an unknown label unchanged, so fidelity looks flawless.

Only the presentation half notices. Had the metric been fidelity alone, this agent would have scored full marks while emitting sentences that are not in the requested language.

In [ ]:
faithful_but_english = 'Elke giraf is een Herbivore.'
axiom = ch9.SAMPLE_AXIOMS[0]
recovered = ch9.parse_cnl(faithful_but_english, 'nl')
print('sentence  :', faithful_but_english)
print('round-trip:', recovered.key() == axiom.key())
print('is Dutch  :', ch9.uses_lexicon_labels(faithful_but_english, axiom, 'nl')['ok'])
print('\nA metric made only of the exact half would have called this perfect.')

## 4. Optimal stopping: when to stop resampling

The agent drafts a verbalisation, sees its quality, and decides: accept, or pay to draw again? This is **best-of-n sampling**, and it is a decision problem with an exact answer.

| | |
|---|---|
| **S** | attempts spent, and the best quality in hand |
| **A** | `accept` the current draft, or `retry` |
| **T** | **stochastic** — a fresh draft's quality is drawn, not chosen |
| **R** | `-cost` per attempt; on accept, the quality accepted |

In [ ]:
M = AG.RevisionMDP(qualities=(0.4, 0.7, 1.0), probabilities=(0.5, 0.3, 0.2),
                   cost=0.05, max_attempts=4)
print('quality ladder :', M.qualities)
print('draw probability:', M.probabilities)
print('cost per attempt:', M.cost, ' budget:', M.max_attempts)
V, pi = mdp.value_iteration(M)
print(f'\nV*(s0) = {V[M.initial_state()]:.4f}')
print(f'expected quality of a single draw = '
      f'{sum(q * p for q, p in zip(M.qualities, M.probabilities)):.3f}')

### The stopping rule, derived:

In [ ]:
print(pd.DataFrame(M.thresholds(pi)).to_string(index=False))

> **Read the table by row.** With attempts to spare, the agent rejects anything below the top quality and draws again. On the **last** attempt it accepts whatever it holds, because there is nothing left to trade.

That is the classic optimal-stopping shape: a **threshold that falls as the budget runs out**. Nobody encoded it — value iteration derived it from the cost and the distribution. If you have ever run best-of-n sampling and guessed at `n`, this is the arithmetic you were guessing at.

In [ ]:
import random
random.seed(0)
rows = []
for cost in [0.0, 0.05, 0.15, 0.3, 0.5]:
    Mc = AG.RevisionMDP(cost=cost)
    Vc, pic = mdp.value_iteration(Mc)
    lengths = [len(mdp.run_episode(Mc, mdp.greedy_policy(pic))) - 1
               for _ in range(300)]
    rows.append({'cost per attempt': cost,
                 'V*': round(Vc[Mc.initial_state()], 4),
                 'mean attempts': round(sum(lengths) / len(lengths), 2)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nFree attempts -> sample until you hit the best draft. Expensive\n'
      'attempts -> take the first thing you get. Everything in between is the\n'
      'interesting case, and it is where real systems live.')

### Task 4.1 — Find the budget beyond which more sampling does not pay

Sweep `max_attempts` and report where `V*` stops improving materially. Explain the shape of the curve.

In [ ]:
rows = []
previous = None
for budget in range(1, 9):
    Mb = AG.RevisionMDP(max_attempts=budget)
    Vb, _ = mdp.value_iteration(Mb)
    value = Vb[Mb.initial_state()]
    rows.append({'max attempts': budget, 'V*': round(value, 4),
                 'gain': '-' if previous is None else round(value - previous, 4)})
    previous = value
print(pd.DataFrame(rows).to_string(index=False))
gains = [r['gain'] for r in rows if r['gain'] != '-']
print('\nDiminishing returns, and quickly: each extra attempt only helps in the\n'
      'worlds where every earlier draw was poor, and those get rarer\n'
      'geometrically. Past a handful of attempts you are paying full price for\n'
      'an increasingly unlikely improvement -- which is why best-of-64 is almost\n'
      'never worth 64 times best-of-1.')

**Checks for Task 4.1.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert gains[-1] < gains[0]

### Task 4.2 — Score fidelity only, and watch the language break

Build a metric that scores **only** the round trip, optimise against it, and show the resulting agent scoring well while producing sentences that are not in the requested language.

In [ ]:
from oe_course.evaluation import ScoreReport

def fidelity_only(gold, pred):
    sentence = str(getattr(pred, 'sentence', '') or '').strip()
    axiom = ch9.Axiom(gold.subject, gold.operator, gold.filler, gold.property)
    recovered = ch9.parse_cnl(sentence, gold.language)
    ok = bool(recovered) and recovered.key() == axiom.key()
    notes = [] if ok else [f'Does not recover {axiom}.']
    violated = [] if ok else [f'template-for-{axiom.operator}']
    return ScoreReport(float(ok), notes, violated)

blind_metric = ev.make_gepa_metric(fidelity_only, AG.CNL_RULEBOOK)
blind = opt.run_gepa(AG.VerbalisationProgram(), train, blind_metric, valset=train,
                     max_metric_calls=140, reflection_lm=reflect)
blind_rules = AG.CNL_RULEBOOK.active_in(opt.instruction_of(blind))
print('rules discovered:', sorted(blind_rules))
print('lexicon rule learned?', 'use-the-lexicon-label' in blind_rules)

fidelity = ev.evaluate_dataset(blind, dev, fidelity_only)['mean_score']
full = ev.evaluate_dataset(blind, dev, AG.verbalisation_scorer)['mean_score']
print(f'\nscored on fidelity only : {fidelity}')
print(f'scored on the full metric: {full}')
print('\nNear-perfect on the metric it was optimised against, and worse on the\n'
      'one that reflects the job. The exact half was never wrong -- it was\n'
      'INCOMPLETE, and an optimiser will find whatever your metric forgot to\n'
      'measure. That is the argument for pairing a decision procedure with a\n'
      'judge rather than choosing between them.')

**Checks for Task 4.2.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert 'use-the-lexicon-label' not in blind_rules
assert full < fidelity

### Task 4.3 — Let the agent grade itself before answering

Show that an agent calling `check_round_trip` can reject its own bad draft without any gold answer, and argue what that changes about deployment.

In [ ]:
ctx2 = AG.Ch9Context()
t2 = {t.name: t for t in AG.build_toolset(ctx2)}
drafts = ['Giraffe subclassof Herbivore',
          'Giraffes are herbivores.',
          'Every giraffe is a herbivore.']
for draft in drafts:
    verdict = json.loads(t2['check_round_trip'].invoke(
        {'sentence': draft, 'language': 'en'}))
    print(f"{str(verdict['parses']):5s} {draft!r:44s} -> {verdict['recovered']}")
accepted = [d for d in drafts
            if json.loads(t2['check_round_trip'].invoke(
                {'sentence': d, 'language': 'en'}))['parses']]
print('\ntool calls:', len(ctx2.log.names()))
print('\nThe agent discarded two of three drafts using only a function it can\n'
      'call. That is worth more than it looks: a self-checkable task can be\n'
      'deployed with a guarantee rather than a hope, and the optimal-stopping\n'
      'MDP above is exactly the policy for using such a check under a budget.')

**Checks for Task 4.3.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert accepted == ['Every giraffe is a herbivore.']

## Chapter 9 in the course arc

| | Ch. 8 | Ch. 9 |
|---|---|---|
| MDP | serve under staleness | **optimal stopping** |
| grader | two paths must agree | **a function and its inverse** |
| what it teaches about metrics | agreement needs no oracle | an exact metric can still be incomplete |

Chapters 8 and 9 make the same point from opposite directions. Chapter 8: the best checks need no gold answer. Chapter 9: even a *perfect* check only measures what it measures. Both are arguments for building the evaluation deliberately rather than reaching for whichever metric is easiest to compute.